In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# **Importing Necessary Libraries**

In [ ]:
# For Visualizing
import seaborn as sns
import matplotlib.pyplot as plt

# For Splitting the Dataset into train and test data
from sklearn.model_selection import train_test_split

# for encoding strategies
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder

# for making pipelines and connecting with models
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# For Imputing purposes
from sklearn.impute import SimpleImputer

# Metrics for computing score and errors
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

# for HPT - Hyper Parameter Tuning the Models
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV


# from sklearn.decomposition import PCA
from scipy.stats import randint , uniform

# To check how does the model performs across various splits
from sklearn.model_selection import cross_val_score

# TO capture non linearity in features and to be used in Linear Models
from sklearn.preprocessing import PolynomialFeatures

# Linear - Based Models 
from sklearn.linear_model import Ridge , Lasso

# Tree Based Models to Capture Non-Linearity and Complexity
from xgboost import XGBRegressor
from xgboost import XGBRFRegressor 
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor # An Extended Version of RF

import warnings
warnings.filterwarnings('ignore')

*XGBRFRegressor is the combination of XGBoost and RandomForest*
*This reduces Overfitting*
*This follows Bagging Algorithm unlike XGBRegressor which uses Boosting Algo*

# **Loading the Train and Test Set**

In [ ]:
# Loading the Train Data set
train_df = pd.read_csv('/kaggle/input/engage-2-value-from-clicks-to-conversions/train_data.csv')

# Loading the Test Set
test_df = pd.read_csv('/kaggle/input/engage-2-value-from-clicks-to-conversions/test_data.csv')
train_df.head()

Determinig The shape/size of the datasets

In [ ]:
print("Shape of train data:" , train_df.shape)
print("Shape of test data:" , test_df.shape)

# **Dummy Project Submission**

In [ ]:
# from sklearn.dummy import DummyRegressor
# dr = DummyRegressor()
# model = dr.fit(X_train,y_train)
# score_value = dr.score(X_test,y_test)

# df_test = pd.read_csv('/kaggle/input/engage-2-value-from-clicks-to-conversions/test_data.csv')
# y_pred = dr.predict(df_test)
# submissions = pd.DataFrame({"id": range(y_pred.shape[0]), "purchaseValue": y_pred})
# submissions.to_csv('submission.csv', index=False)

# gave a score of 0.00

# **EDA - Exploratory Data Analysis**

Exploring Train Data:
- features
- Nullity
- Data Types

In [ ]:
print("The number of Samples (Rows) and Features (Columns) of the Dataset are :", train_df.shape)

In [ ]:
train_df.info()

from the above output we can observe that there are 
- 37 Object Features
- 14 Numerical Features (9 - float , 5 - int)
- 1 boolean feature

out of these features there are 10 features which contains missing values.
Out of these features which have missing rows which will be retianed depends on thier Nullity (For now , we will consider nullity , after wards the idea of keeping of these features shall be decided on how the model performs , thier cardinality).

In [ ]:
train_df.describe()

By running the above cell which describes the features of the data , but on noticing we can see that it only is describing for numerical columns and not for categorical columns.

From this `describe()` method , it tells some vital information :-
1. Target Feature - `purchaseValue` is extremely right skewed. As mean is greater than median and also there is a vast difference between min and max value.
2. `userId` has mean and median almost equal to each other (almost close) , this shows it could be Normal or Uniform distribution , which will be made sure by subsequent visualizations.
3. `gclIdPresent` is a boolean feature.
4. `sessionNumber` seems slight right skewed distribution with outliers present in it.
5. `total.visits` & `new_visits` only includes one unique value - 1 , which acts like binary feature.
6. `sessionId` also is right skewed distribution.
7. `date` feature is int in this data , but it should be in datetime format, so i cant say anything about this feature untill this is convereted to datetime format.
8. `sessionStart` is also same logic as of `date` because it includes timestamps so I cant rely on it now.

Checking if the Target is uniform or if is it skewed

In [ ]:
sns.boxplot(x = train_df['purchaseValue'])

from the graph , it is evident that most of the users have not made any purchases , but in the describe column we can see that mean value of this column is way too huge , that is because the few customers who have made the purchases have made of bigger amounts , also max and min have a huge gap so this shows it includes a extreme outlier present.

In [ ]:
sns.histplot(train_df['purchaseValue'])

again we can see that most are at 0 but there is a tail on the x - axis 

In [ ]:
sns.histplot(np.log1p(train_df['purchaseValue']))

the above graph shows that it is highly Right-Skewed distribution. the purpose of using log1p was to compress the large values and enhance the smaller values to the screen to detect for skewness clearly.

In [ ]:
# Finding the number of people who have not made any purchase 
no_purch = len(np.where(train_df['purchaseValue'] == 0)[0])
print(no_purch)

There are about 92,038 users who did not made any purchases (which is about 79.3%) of the users of all the users. This explains the reason for skeweness.

from the `.info()` it shows clearly that the feature - `date` is a int dtype , but it should be in `datetime` dtype so I transformed it from int to datetime.

In [ ]:
train_df['date'] = pd.to_datetime(train_df['date'] , format = '%Y%m%d')

In [ ]:
train_df['date']

Checking the Range og the Data Collection Period

In [ ]:
print(f"First Date:{train_df['date'].min()}")
print(f"Last Date:{train_df['date'].max()}")

This shows the date collection period from `01/08/2016` to `30/04/2018`

In [ ]:
# Seperating Categorical and Numerical cols
categorical_cols = []
numerical_cols = []

for i in train_df.columns:
        if train_df[i].dtype == 'object' or train_df[i].dtype == 'bool':
            categorical_cols.append(i)
        elif train_df[i].dtype == 'int64' or train_df[i].dtype == 'float64':
            numerical_cols.append(i)

In [ ]:
print(f"Number of Numerical columns are: {len(numerical_cols)}")
print(f"Number of Categorical columns are: {len(categorical_cols)}")

In [ ]:
numerical_cols

Finding which year has the highest purchases value.

In [ ]:
train_df['year'] = train_df['date'].dt.year
yearly_purch = train_df.groupby('year')['purchaseValue'].sum()
yearly_purch

it is 2017 which made the highest purchase values.

Creating a df to store the users and thier details who all made a purchase.

In [ ]:
# Finding the Number of Users who have made a Purchase
purch_users = train_df[train_df['purchaseValue'] > 0]
purch_users

from the above output it shows that around 23,985 users have made the purchase while the other have not , this constitutes about 20.6% of users who have made a purchase

In [ ]:
# Finding in Which Year highest purchases have been made
purch_users.groupby('year')['userId'].nunique()
# Finding if 2017 year has the most number of users who have made purchases
# or was it some other year with highest number of customer engagement

It proves that in 2017 , it has the highest customer engagement comapred to other years.

In [ ]:
for i in train_df.columns:
    if train_df[i].nunique() == 1:
        print(f"Col Name: {i} , Data Type: {train_df[i].dtype}")
        print(f" Unique Values : {train_df[i].unique()}")
        print(f"No Of Null rows: {train_df[i].isna().sum()}")
        print("_________________________________________")

from the above output , there are a lot of features which have only one value throughout the whole feature (i.e, no of unique value is just 1) with no missing rows --> For these features it is evident / viable to remove them as these features are not adding any meaning or giving any valuable info in predicting the output value.

Apart from these features , we have few other features which also have only one unique value but they also have missing rows present in them , like the feature 

`TrafficSource.isDirect` has about 63% of rows missing 
`trafficSource.adwordsClickInfo.isVideoAd` has more than or equal to 90% of rows missing - we can remove this feature as Imputing this will also almost make this one valuer.
`totals.bounces` has about 59% of rows missing 
`new_visits` also has about 30% of rows missing which is the lowest among above features , so we can easily impute this column.
`

## **Analysing Distribution (nature) of the features**

### **Analysing Numerical Feaures**

In [ ]:
import matplotlib.pyplot as plt

for col in numerical_cols:
    plt.figure(figsize=(14, 5))
    sns.histplot(train_df , x = col , kde = True , bins = 100)
    plt.title(f'Histplot of {col}')

    plt.show()

#### **Observations from the numerical features histplot**

Above I have plotted histplot of Numerical features to study them

1. `purchaseValue` - As it is already seen that this graph is highly positive skewed (Right Skewed) with outliers present in it.

2. `userId` - The Histplot is approximate to a uniform distribution , where almost each has same frequency. here in this graph the sessions of a user is almost equal to any other user's number of sessions.

3. `gclIdpresent` - this is binary feature consisting of only 0s and 1s , where the frequency of os is greater than 1s.

4. `sessionNumber` - this feature's histplot shows similar properties to target feature , which is highly right skewed (Positive skewed) distribution. The shape indicates that the vast majority of sessions are from users with a low session number. As the session number increases, the count drops dramatically, meaning very few users return for a large number of sessions.

5. `total_visits` , `locationZone` , `total.bounces` and `new_visits` - These features are constant features meaning they have only one unique value and 0 variance, they does not contribute much to prediction process.

6. `sessionstart` and `sessionId` - These feature's histplot shows similar properties which shows that they have high correlation between them and both are multimodal, here we can either remove one of them or do some feature engg on one of them. As `sessionStart` is a timestamp feature so we can extract hour , weekend etc out of it.

### **Analysing Categorical Features**

In [ ]:
for col in categorical_cols:
    plt.figure(figsize = (14, 5))
    sns.countplot(data = train_df , x = col)
    plt.title(f"Countplot of {col}")
    plt.show()

#### **Observations from above plots for categorical fetures**

From the above plots for categorical , here is what I have observed:
1. Constant Features - There are about 15+ constant features which only consists of only one unique category , as they does not provide much to our predictive process so they needs to be removed so that the model does not learns noise from them.

2. Uniform Distributions - There are about 2 features which follow uniform distribution - `geoCluster` and `geoNetwork.networkDomain` which means they doesn’t have a 
dominant range where one class differs from another

3. High Cardinality features - There are features (like - `trafficSource.isreferralPath` (942) , `geoNetwork.city` (695) and `trafficSource.keyword` (566)) which have very high cardinality , means these features have very high number of categories , dealing with these high cardinality features may takes more time in computation while model training and in while HPT (Hyper-Parameter Tuning). Models may struggle to generalize well on rare/unseen categories, reducing robustness.

4. Missing Rows - There are also some features which includes missing rows to a great extent like in some features 90% or more missing content is missing , so removing them is a viable option here as 

In [ ]:
# Plotting the heatmap
train_df_copy = train_df[numerical_cols]
corr_matrix = train_df_copy.corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', square=True)
plt.title("Correlation Heatmap of Numerical Features")
plt.show()

**Key Observations from heat map**

1. Multicollinearated featurees - features like 'pageViews' and 'totalHits' are highly correlated with each other (a perfect correlation - 0.99) , simiarly features like 'sessionId' and 'sessionStart' are also highly correlated with each other. either one of these features need to be removed or grouping them into feature is required to reduce the effect multi - collinearity.
2. Empty spaces - there are empty spaces within heatmap , which might be because these features are either constant or have missing rows to a great extent , that's why it was not able to calculate the correlation.
3. Higher predictor - features like 'sessionNumber' , 'pageViews' and 'totalHits' are higher predictors for our target variable , so it is necessary to retian them.

# **Pre - Processing**

so in pre - processing we will start with removing the features which have only one cardinality and the other features which have more than 90% missing values , and we shall identify them by running a loop and then drop them as they contain no meaningful information and does not help in predicting towards the target value.

Checking the features in test dataframe for any dis-similarity in features names.

In [ ]:
test_df.info()

In [ ]:
cols_to_drop = []

for cols in train_df.columns:
  if train_df[cols].nunique() == 1 or (train_df[cols].isna().sum() / train_df.shape[0]) >= 0.6:
    print(cols)
    print("Number of Unique Values:" , train_df[cols].nunique())
    print("% of Missing rows:" , (train_df[cols].isna().sum() / train_df.shape[0]))
    print("Data Type:" , train_df[cols].dtype)
    cols_to_drop.append(cols)
    print('__________________________________________________________________')

In [ ]:
cols_to_drop

In [ ]:
train_df.drop(columns = cols_to_drop , inplace = True)
test_df.drop(columns = cols_to_drop , inplace = True)

On observing the features which contain missing values , there is a feature - `pageViews` which in Analytics means that on clicking the Ad how many pages did user visit to , so if it has missing rows that could possibly mean that user did not visit there , so we shall impute it with 0.

In [ ]:
train_df['pageViews'] = train_df['pageViews'].fillna(0)
test_df['pageViews'] = test_df['pageViews'].fillna(0)

Feature - `device.isMobile` is a boolean which means it consists of True/False with no missing values so for a ML Model we need to convert this to numerical values , here we shall encode with binary values -> 0 for False and 1 for True.
This feature basically tells that user used a mobile phone to view the ad or not.

In [ ]:
train_df['device.isMobile'] = train_df['device.isMobile'].astype(int)
test_df['device.isMobile'] = test_df['device.isMobile'].astype(int)

In [ ]:
train_df.drop(columns = ['geoNetwork.networkDomain' , 'geoCluster' ] , inplace = True) 
test_df.drop(columns = ['geoNetwork.networkDomain' , 'geoCluster' ] , inplace = True)

Now features - `trafficSource.referralPath` and `trafficSource.keyword` consists of more than 60% missing rows but the problem with these features is that they are of high cardinality means that they have a large number of Unique values - 566 and 942.

The column `trafficSource.keyword` contains approximately 72,000 missing (empty) values.
Additionally, when inspecting the unique values using `value_counts()`, it was observed
that the most frequent value was `(not provided)` — appearing 39,066 times.

If we combine the missing values (71,861) and the `(not provided)` entries (39,066),
we get a total of 110,927 rows — which is about 95.6% of the entire dataset.
This indicates that the column lacks informative content for the vast majority of records
and offers very limited variance or signal for predicting the target variable.

Hence, the feature `trafficSource.keyword` was dropped from the dataset.

The columns `userId` and `sessionId` were dropped because they serve as unique identifiers rather than predictive features. They do not carry any meaningful pattern or relationship with the target variable, and including non-informative identifiers can introduce noise and increase the risk of overfitting without improving model performance.

In [ ]:
train_df.drop(columns = ['sessionId'] , inplace = True)
test_df.drop(columns = ['sessionId'], inplace = True)

In [ ]:
train_df.info()

# **Splitting the training data into X and y**

In [ ]:
X = train_df.drop(columns = 'purchaseValue')
y = train_df['purchaseValue']

# **Feature Engg**

The `date` column was originally in integer format (values like 20120923), which made it difficult to interpret and use directly in modeling. To make it meaningful, the column was first converted to a proper datetime format using pandas. From this converted date, we extracted the year and month components as separate features. This allows the model to understand and leverage time-based patterns, such as seasonality, yearly trends, or monthly spikes in activity, which may influence user behavior or purchase value.

In [ ]:
X['month'] = X['date'].dt.month

test_df['date'] = pd.to_datetime(test_df['date'] , format = '%Y%m%d')
test_df['year'] = test_df['date'].dt.year
test_df['month'] = test_df['date'].dt.month

The `sessionStart` column, which stores session timestamps in Unix time format (seconds since epoch), was first converted into a readable datetime format using pandas. From this, three new features were derived:

- `session_hour` captures the hour of the day the session began, which helps in identifying time-based user activity (e.g., active hours vs. idle hours).

- `session_dayofweek` extracts the day of the week (0 = Monday, 6 = Sunday), allowing the model to detect weekly usage trends.

- `is_weekend` is a binary feature indicating whether the session occurred on a weekend (Saturday or Sunday). User behavior often differs between weekdays and weekends, so this feature may improve the model’s ability to capture such patterns.

In [ ]:
X['session_hour'] = pd.to_datetime(X['sessionStart'], unit = 's').dt.hour
# Here unit='s' is specified because the time is in seconds 
test_df['session_hour'] = pd.to_datetime(test_df['sessionStart'], unit = 's').dt.hour

X['session_dayofweek'] = pd.to_datetime(X['sessionStart'], unit = 's').dt.dayofweek
test_df['session_dayofweek'] = pd.to_datetime(train_df['sessionStart'], unit = 's').dt.dayofweek

X['is_weekend'] = X['session_dayofweek'].isin([5, 6]).astype(int)
test_df['is_weekend'] = test_df['session_dayofweek'].isin([5, 6]).astype(int)

In [ ]:
X.drop(columns = ['date' , 'sessionStart'] , inplace = True)
test_df.drop(columns = ['date' , 'sessionStart'] , inplace = True)

Seperating numerical features and categorical features into different lists.

In [ ]:
num_cols = []

for i in X.columns:
    if X[i].dtype != 'object':
        num_cols.append(i)

len(num_cols)

In [ ]:
num_cols

In [ ]:
cat_cols = []
grouping_cols = []

for i in X.columns:
  if i not in num_cols:
      cat_cols.append(i)
      print(i , X[i].nunique())
      if X[i].nunique() > 100:
        grouping_cols.append(i)

len(cat_cols)

In [ ]:
cat_cols

In [ ]:
def cumulative_sum(X , col_name):

  vc_pct = X[col_name].value_counts(normalize = True) * 100
  cum_sum  = 0
  cumsum_map = {}

  for category , percentage in vc_pct.items():
    cum_sum += percentage
    cumsum_map[category] = round(cum_sum,2)

  # cumsum_map returns a dict with category names as keys
  # and cum sum values as values of the dictionary
  # We can modify this dict to a Pandas Series for easy access

  cumsum_series = pd.Series(cumsum_map)

  return cumsum_series

As it has been observed that many features in the dataset are poorly structured with some having high cardinality, some being highly imbalanced, and some showing both issues so it becomes important to group or simplify these features.

To address this, I developed a function that helps retain only the most informative categories from a feature, based on their frequency distribution.

The function supports two approaches:

- Retaining a fixed number of top categories (top_n)

- Retaining enough categories to cover a certain percentage of total data (coverage_threshold)

This helps reduce noise, improve model interpretability, and handle rare or irrelevant categories effectively.

In [ ]:
def get_top_categories(X , col_name , coverage_thresh , top_n):

    # coverage_threhold is the (%) value that we will retain 
    # Like if we want to retain 80% of feature's information
    # Then how many categories does it need to retain given % of information.

    # top_n is the fixed number of categories we need to retain
    # like if we want to retain top 10 categories then it would simply retain 
    # top 10 categories from a given feature which are important and holds
    # most of the information for a particular feature. 

    if top_n is not None and coverage_thresh is not None:
        return 'Not valid , give only one of the params'
    
    value_pct = X[col_name].value_counts(normalize = True) * 100
    # Storing the values as a percentge which I will use these values
    # in finding the req number of features based on their cumulative sum

    # Calculating Cumulative Sum 
    value_cumsum = value_pct.cumsum()
    # Now this could have been done using via Pandas.
    # Also using either cumsum() or the function created returns a 
    # Pandas Series which helps to access the values less or greater than a thresh

    if top_n is not None: # If value is given for top_n
        top_categories = value_pct.index[:top_n].to_list()
        # So here we are accessing top N categories specified while calling the function
        # and then storing them as a list to later visulaize them
        print("Retaining top" ,top_n, "categories (fixed).")
        return top_n, top_categories

    elif coverage_thresh is not None:
        n_categories = (value_cumsum <= coverage_thresh).sum() + 1
        # we are adding this condition to return number of categories 
        # which are less than thresh value as it returns a boolean series of T/F
        # so adding them gives a count of True values
        top_categories = value_pct.index[:n_categories].tolist()
        print("To cover at least", coverage_thresh , "%", "retain top" , n_categories,  "categories.")
        return n_categories, top_categories

    else:
        return "Provide atleast value for one of the parametrs: top_n or coverage_threshold"

In [ ]:
def grouping_features(X , test_df , col_name , top_categories):

  grouped_col_train = [] # To store the grouped values

  for cats in X[col_name]:
    if cats not in top_categories:
      grouped_col_train.append('Others')
    else:
      grouped_col_train.append(cats)

  X[col_name] = grouped_col_train

  grouped_col_test = [] # To store the grouped values

  for cats in test_df[col_name]:
    if cats not in top_categories:
      grouped_col_test.append('Others')
    else:
      grouped_col_test.append(cats)
      
  test_df[col_name] = grouped_col_test

  return X[col_name].nunique()

In [ ]:
for col in grouping_cols:
    print(f"Processing: {col}")
    
    n_categories, top_categories = get_top_categories(X, col, coverage_thresh = 95 , top_n = None)
    
    final_unique = grouping_features(X, test_df, col, top_categories)
    
    print(f"After grouping, number of unique categories in '{col}': {final_unique}")
    print('-------------Line Break-----------------------')

In [ ]:
low_card_cols , high_card_cols = [] , []

for col in cat_cols:
  print(col,":", "Cardinality:", X[col].nunique())
  print("-------------------")
  if X[col].nunique() > 10:
    high_card_cols.append(col)
  else:
    low_card_cols.append(col)

In [ ]:
X.info()

# **Encoding Categorical Features**

In [ ]:
def frequency_encoding(X, cols_list):
    
    X_encoded = X.copy()
    freq_mappings = {}

    for col in cols_list:
        freqs = X_encoded[col].value_counts(normalize = True).to_dict()
        X_encoded[col] = X_encoded[col].map(freqs).fillna(0)
        freq_mappings[col] = freqs  

    return X_encoded, freq_mappings

In [ ]:
def frequency_encoding_test(X , freq_mappings):
    # We will use freq mappings from train_data 
    # to encode categories in test_data
    X_encoded = X.copy()

    for col, freqs in freq_mappings.items():
        X_encoded[col] = X_encoded[col].map(freqs).fillna(0)

    return X_encoded

In [ ]:
def label_encode_inplace(X, low_card_cols):

    # Dictionary to store the fitted encoders for each feature
    encoders = {}

    # Looping through each column in the list of low-cardinality columns
    for col in low_card_cols:
        le = LabelEncoder()

        # Convert values to string — avoids issues when data has mixed types
        # For example, if some values are strings and others are numbers
        X[col] = X[col].astype(str)

        # Fit the LabelEncoder on this column’s values
        # It learns unique categories and assigns them integer labels
        # E.g., ['mobile', 'desktop', 'tablet'] → [1, 0, 2]
        X[col] = le.fit_transform(X[col])

        # Saving the fitted LabelEncoder object in the dictionary
        # This can be reused to:
        # - encode test/unseen data
        # - reverse the encoding via le.inverse_transform()
        encoders[col] = le

    # Return the dictionary of all fitted encoders
    return encoders

In [ ]:
def label_enc_test(test_df , encoders):

    for col , le in encoders.items():
        # First we convert cols of test_df to str
        # to avoid any errors in encoding process
        test_df[col] = test_df[col].astype(str)

        encoded_values = []
        # this list will store the encoded values
        # for each category in a col for test_df
        # and then apply that to test_df[col]
        for val in test_df[col]:
            # if the category exists in test_df 
            # for a particular col then we will
            # encode that with value learned during 
            # encoding process in train_df
            if val in le.classes_:
                encoded_val = le.transform([val])[0]

            else:
                encoded_val = -1
            # Now we shall append the encoded values
            # for each category in test_df[col]
            # to encoded_values list
            encoded_values.append(encoded_val)

        test_df[col] = encoded_values  # Assigning encoded values

    return test_df

In [ ]:
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size = 0.10 , train_size = 0.90 , random_state = 42 , shuffle = True)

In [ ]:
X_train, freq_mappings = frequency_encoding(X_train, high_card_cols)
X_test = frequency_encoding_test(X_test, freq_mappings)

test_df = frequency_encoding_test(test_df, freq_mappings)

In [ ]:
df = X_train.copy()
df['purchaseValue'] = y_train

user_avg_purchase = df.groupby('userId')['purchaseValue'].mean().reset_index()
user_avg_purchase.columns = ['userId', 'avg_purchase_per_user']

X_train = X_train.merge(user_avg_purchase, on='userId', how='left')
X_test = X_test.merge(user_avg_purchase, on='userId', how='left')

X_test['avg_purchase_per_user'] = X_test['avg_purchase_per_user'].fillna(df['purchaseValue'].mean())

test_df = test_df.merge(user_avg_purchase, on='userId', how='left')

# Handle unknown users (users in test_df but not in training data)
test_df['avg_purchase_per_user'] = test_df['avg_purchase_per_user'].fillna(df['purchaseValue'].mean())

In [ ]:
test_df.info()

# **Setting of Pipelines**

In [ ]:
num_pipeline = Pipeline([
    ('num' , StandardScaler())
])

cat_pipeline = Pipeline([
    ('cat' , OneHotEncoder(handle_unknown = 'ignore' , sparse_output = False , drop = 'first'))
])

ct = ColumnTransformer([
    ('num_cols', num_pipeline, num_cols), 
    ('cat_cols' , cat_pipeline, low_card_cols)
], remainder='passthrough')

ct.fit(X)

# **Plotting Heatmap**

In [ ]:
scaled_num_data = ct.named_transformers_['num_cols'].transform(X[num_cols])
scaled_num_df = pd.DataFrame(scaled_num_data, columns = num_cols, index = X.index)

scaled_num_df['target'] = y.values

df_corr = scaled_num_df.corr()

plt.figure(figsize = (20, 15))
sns.heatmap(df_corr, annot = True, fmt = ".2f", cmap = 'coolwarm', square = True)
plt.title('Correlation Heatmap (Scaled Numerical Features + Target)')
plt.show()

# **Model Training & Tuning**

Checking for Complexity Level by Training the Model via Linear Models
- `Ridge()`
- `Lasso()`

Training via Linear Model - `Ridge()`

In [ ]:
ridge_pipeline = Pipeline([
    ('preprocessor' , ct),
    ('ridge' , Ridge(random_state = 42))
])

ridge_pipeline_model = ridge_pipeline.fit(X_train , y_train)

Here `include_bias = False` means that we are not adding the cols of which is bias (intercept) column, that is because in model like - `Ridge()` & `Lasso()` they add this feature automatically (internally)

In [ ]:
y_train_pred = ridge_pipeline_model.predict(X_train)
y_test_pred = ridge_pipeline_model.predict(X_test)

print("Train Set:")
print("R2 Score        :", r2_score(y_train, y_train_pred))

print("\n Test Set:")
print("R2 Score        :", r2_score(y_test, y_test_pred))

From the above observations, it is evident that the default `Ridge()` model (without Polynomial Transformation) performs better in terms of generalization, as it shows minimal overfitting.

However, when we introduced a PolynomialFeatures transformation, the model started to overfit heavily but despite this, both MAE and RMSE significantly decreased, indicating that the model fit the training data more closely, even though it failed to generalize well on unseen data.

In [ ]:
# param_grid_ridge = {
#     'ridge__alpha': list(np.random.randint(1000, 2000, size=100)) 
#     # Here it will take 100 random values in the range of 1000 to 2000
# }

# grid_search_ridge = RandomizedSearchCV(
#     ridge_pipeline,
#     param_grid_ridge,
#     n_iter = 100,
#     cv = 5,
#     scoring = 'r2',
#     n_jobs = -1,
#     random_state = 42
# )

# grid_search_ridge.fit(X_train, y_train)

# 100 sec

In [ ]:
print("Best parameters:", grid_search_ridge.best_params_)

# Evaluating on test set
y_test_pred = grid_search_ridge.predict(X_test)
print("Test R2 Score:", r2_score(y_test, y_test_pred))

After fine-tuning the regularization strength (alpha) in the range of 1000 to 2000, the best performance was achieved with alpha = 1999. However, both the cross-validation $R^2$ (≈ 0.0975) and test $R^2$ (≈ 0.1942) showed only marginal improvement compared to the earlier default Ridge model. The error metrics (MAE and RMSE) also saw slight reductions, indicating a slightly better fit but no significant gain in generalization. This suggests that a linear model, even when finely tuned, may be inherently limited for this dataset, and exploring non-linear models like Random Forest or XGBoost could yield better performance. But before going to Non-Linear models , we would explore another linear model - `Lasso()` which might perform better than Ridge.

Training via `Lasso()`

In [ ]:
lasso_pipeline = Pipeline([
    ('preprocessor' , ct),
    ('poly_trans' , PolynomialFeatures(degree = 2 , include_bias = False)),
    ('lasso' , Lasso(random_state = 42))
])

lasso_pipeline_model = lasso_pipeline.fit(X_train , y_train)

# 120 sec

In [ ]:
# Predictions on Test Data
y_train_pred = lasso_pipeline_model.predict(X_train)
y_test_pred = lasso_pipeline_model.predict(X_test)

# Calculating Score & Error Metrics
print("Train Set:")
print("R2 Score        :", r2_score(y_train, y_train_pred))

print("\nTest Set:")
print("R2 Score        :", r2_score(y_test, y_test_pred))

In [ ]:
param_grid_lasso = {
    'poly_trans__degree' : [1],
    'lasso__alpha': [1, 10, 100 , 1000],
    'lasso__max_iter': randint(2000 , 3000)
}

random_lasso = RandomizedSearchCV(
    lasso_pipeline,
    param_grid_lasso,
    scoring = 'r2',
    n_iter = 5,
    cv = 3,
    n_jobs = -1,
    random_state = 42
)

random_lasso.fit(X_train, y_train)

# Took around 227 sec

In [ ]:
print("Best alpha:", random_lasso.best_params_['lasso__alpha'])
print("Best Degree:", random_lasso.best_params_['poly_trans__degree'])
print("Best Iter Value:", random_lasso.best_params_['lasso__max_iter'])
print("CV Best R2:", random_lasso.best_score_)

y_pred = random_lasso.predict(X_test)
print("Test R2 Score:", r2_score(y_test, y_pred))
print("Test MAE:", mean_absolute_error(y_test, y_pred))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

from the above observations , `Lasso()` performed better without Hyper-Parameter Tuning where only one degree for PolyTransformation is given i.e, degree = 1 , because of this the model was not able to capture the non - linear essence and hence it performed worse. Whereas when trained without hyper parameter with poly degree = 2 , it gave a better score but it is underfitting which might be because Even after using degree = 2, the model was still underfitting because the default alpha value (regularization strength) might have been too high. This forced the model to shrink most coefficients close to zero, reducing its ability to learn from the new polynomial features. Also, without hyperparameter tuning, the model couldn't adjust properly to the increased feature complexity, so it failed to capture the patterns fully.

- **XGBoost Regressor**
    - Model Training on --> `XGBRegressor()`
    - Analysing Feature Importance
    - Peforming HP (Hyper - Tuning) & checking its Preformance

In [ ]:
xgbr_pipeline = Pipeline([
    ('preprocessor', ct),
    ('xgbr_reg', XGBRegressor(random_state=42, n_jobs=-1, tree_method = 'hist'))
])

xgbr_pipeline_model = xgbr_pipeline.fit(X_train, y_train)

y_pred_test = xgbr_pipeline_model.predict(X_test)
y_pred_train = xgbr_pipeline_model.predict(X_train)

print("Test R2:", r2_score(y_test, y_pred_test))
print("Train R2:", r2_score(y_train, y_pred_train))

**Feature Importance of XGBoost Regressor**

In [ ]:
# Extracting feature names after preprocessing
feature_names = xgbr_pipeline_model.named_steps['preprocessor'].get_feature_names_out()

# Getting trained XGBRegressor model from pipeline
model = xgbr_pipeline_model.named_steps['xgbr_reg']

# Extracting feature importances
importances = model.feature_importances_

# Combining into a DataFrame
feat_imp_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by = 'Importance', ascending = False)

feat_imp_df

**XGBoost --> Hyper - Parameter Tuning**

In [ ]:
param_dist_xg = {
    "xgbr_reg__eta" : uniform(0.005 , 0.3),
    "xgbr_reg__gamma": randint(5 , 15),
    "xgbr_reg__max_depth" : randint(1, 30),
    "xgbr_reg__subsample" : uniform(0.5 , 0.5),
    "xgbr_reg__min_child_weight" : randint(3,20),
    "xgbr_reg__colsample_bytree" : uniform(0.5,0.5),
    "xgbr_reg__lambda" : randint(5 , 20),
    "xgbr_reg__alpha" : randint(1, 10)
}

xgbr_search = RandomizedSearchCV(
    estimator = xgbr_pipeline,
    param_distributions = param_dist_xg,
    n_iter = 50,
    cv = 5,
    n_jobs = -1,
    scoring = 'r2',
    random_state = 42
)

# Fitting the data
xgbr_search.fit(X_train, y_train)

# Training the model with parameters obtained via HPT
best_xgboost = xgbr_search.best_estimator_

y_pred_test = best_xgboost.predict(X_test)
y_pred_train = best_xgboost.predict(X_train)

print("Best Params:", xgbr_search.best_params_)
print("Test R2:", r2_score(y_test, y_pred_test))
print("Train R2:", r2_score(y_train, y_pred_train))

# Takes around 14 mins

XGBoost gave a decent test score without tuning, but after applying hyperparameter tuning, its performance improved significantly (test R2 Score from 0.53 -> 0.59). This shows that XGBoost is quite sensitive to its parameters, and tuning helped it generalize better.

However, overfitting was still present both before and after tuning, as seen from the large gap between train and test R2 Scores. A simple reason for this could be that the model is very powerful and tends to memorize the training data unless carefully regularized or exposed to more diverse data.

In [ ]:
# test_preds_xgboost = best_xgboost.predict(test_df)
# test_preds_xgboost = np.clip(test_preds_xgboost, 0, None)
# print(test_preds_xgboost)

In [ ]:
# submission = pd.DataFrame({
#     'id': range(len(test_preds_xgboost)),  
#     'purchaseValue': test_preds_xgboost
# })

# # Save to CSV for submission
# submission.to_csv("submission.csv", index = False)

# # output in Kaggle notebook
# print("Submission file created successfully!")
# print(submission.head(10))

- **Random Forest Regressor**
    - Model Training on --> `RandomForestRegressor()`
    - Peforming HP (Hyper - Tuning) & checking its Preformance

In [ ]:
rf_pipeline = Pipeline([
    ('preprocessor', ct),
    ('rf_reg', RandomForestRegressor(random_state = 42, n_jobs = -1))
])

rf_pipeline_model = rf_pipeline.fit(X_train, y_train)

y_pred_test = rf_pipeline_model.predict(X_test)
y_pred_train = rf_pipeline_model.predict(X_train)

print("Test R2:", r2_score(y_test, y_pred_test))
print("Train R2:", r2_score(y_train, y_pred_train))

Till here i have noticed that Random Forest model (without and with hyper-parameter tuning) is learning the data too well , it has the highest highest score on training data (X_train) 
BUT
it comes up with negtive points:
- moderate score on test score 
- high level of Overfitting (covering almost a diff of ~ 0.488)

This model can perform well :
- in case of HPT (Hyper Parameters Tuning) Parameters


**Feature Importance for Random Forest Regressor**

In [ ]:
rf_model = rf_pipeline_model.named_steps['rf_reg']
feature_names = ct.get_feature_names_out()

importances = rf_model.feature_importances_

feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by = 'Importance', ascending = False)

print(feature_importance_df)

**Hyper Paramter Tuning for Random Forest Regressor**

In [ ]:
param_dist_rf = {
    'rf_reg__n_estimators': randint(100, 300),    
    'rf_reg__max_depth': randint(2, 30),          
    'rf_reg__min_samples_split': randint(5, 30),  
    'rf_reg__min_samples_leaf': randint(2, 20),   
    'rf_reg__max_features': ['sqrt', 'log2', 0.5, 0.7], 
    'rf_reg__ccp_alpha': uniform(0.0001, 0.02) 
}

rf_search = RandomizedSearchCV(
    estimator = rf_pipeline,
    param_distributions = param_dist_rf,
    n_iter = 10,
    cv = 3,
    n_jobs = -1,
    scoring = 'r2',
    random_state = 42
)

rf_search.fit(X_train, y_train)

# Using the best estimators obtained from HPT of the abve code
best_rf = rf_search.best_estimator_

y_pred_test = best_rf.predict(X_test)
y_pred_train = best_rf.predict(X_train)

print("Best Params:", rf_search.best_params_)
print("Test R2:", r2_score(y_test, y_pred_test))
print("Train R2:", r2_score(y_train, y_pred_train))

# 446 sec

In [ ]:
# test_df = test_df[X.columns] # To Prevent the Order as X
# test_preds_rf = rf_pipeline_model.predict(test_df)
# # test_preds_etr = np.clip(test_preds_etr, 0, None)
# print(test_preds_rf)

In [ ]:
# submission = pd.DataFrame({
#     'id': range(len(test_preds_rf)),  
#     'purchaseValue': test_preds_rf
# })

# # Save to CSV for submission
# submission.to_csv("submission.csv", index = False)

# # output in Kaggle notebook
# print("Submission file created successfully!")
# print(submission.head(10))

- Random Forest performed strongly without tuning, with a test R2 of 0.58 and a high train R2 of 0.93, indicating some degree of overfitting, as the model was learning the training data very well.

- After applying hyperparameter tuning, the model showed more balanced scores on internal splits (train R2 = 0.65, test R2 = 0.55), suggesting that tuning helped reduce overfitting.

**XGBoost + Random Forest Regressor**
- Model Training on --> `XGBRFRegressor()`
- Checking Feature Importance
- Tuning Parameters and choosing the best params

In [ ]:
xgbrf_pipeline = Pipeline([
    ('preprocessor', ct),
    ('xgbrf_reg', XGBRFRegressor(                   
        random_state=42,
        n_jobs=-1
    ))
])

xgbrf_pipeline_model = xgbrf_pipeline.fit(X_train , y_train)

In [ ]:
y_pred_test_xgbrf = xgbrf_pipeline_model.predict(X_test)
r2_test = r2_score(y_test, y_pred_test_xgbrf)
print("R2 Score on Test Set:", r2_test)

# Predicting on Train Set
y_pred_train_xgbrf = xgbrf_pipeline_model.predict(X_train)
r2_train = r2_score(y_train, y_pred_train_xgbrf)
print("R2 Score on Train Set:", r2_train)

**Feature Importance from XGBoost + RF**

In [ ]:
xgbrf_model = xgbrf_pipeline_model.named_steps['xgbrf_reg']

feature_names = xgbrf_pipeline_model.named_steps['preprocessor'].get_feature_names_out()

importances_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': xgbrf_model.feature_importances_
})

importances_df = importances_df.sort_values(by='Importance', ascending=False)

print(importances_df)

**Hyper-Parameter Tuning for XGBoost + RF**

In [ ]:
param_dist = {
    'xgbrf_reg__n_estimators': [100, 200, 300],  
    'xgbrf_reg__max_depth': [2, 3, 4, 5 , 6],  
    'xgbrf_reg__subsample': [0.5, 0.6, 0.7],  
    'xgbrf_reg__colsample_bytree': [0.6, 0.7 , 0.8], 
    'xgbrf_reg__gamma': [1, 3, 5],  
    'xgbrf_reg__reg_alpha': [5, 10 , 20], 
    'xgbrf_reg__reg_lambda': [10, 15, 20, 50, 100] 
}

random_search1 = RandomizedSearchCV(
    xgbrf_pipeline,
    param_distributions=param_dist,
    n_iter = 10,
    scoring ='r2',
    cv = 3,
    n_jobs = -1,
    random_state = 42
)

random_search1.fit(X_train, y_train)

In [ ]:
Best model
best_xgbrf = random_search1.best_estimator_

# Train & Test R²
r2_train = r2_score(y_train, best_xgbrf.predict(X_train))
r2_test = r2_score(y_test, best_xgbrf.predict(X_test))

print("Best Params:", random_search1.best_params_)
print("Train R2:", r2_train)
print("Test R2:", r2_test)

The XGBRFRegressor underfit the data, as seen from its low and close R2 scores on both train (~0.50) and test (~0.47) sets. This happened because XGBRF is designed to be more conservative by combining ideas from Random Forest and XGBoost as it builds trees on random subsets of data


In [ ]:
etr_pipeline = Pipeline([
    ('preprocessor' , ct),
    ('et_reg' , ExtraTreesRegressor(random_state = 42 , n_jobs = -1 , bootstrap = True, criterion = 'squared_error' , max_features = 'sqrt'))
])

etr_pipeline_model = etr_pipeline.fit(X_train , y_train)

train_pred = etr_pipeline_model.predict(X_train)
test_pred = etr_pipeline_model.predict(X_test)

train_score_etr = r2_score(y_train , train_pred)
test_score_etr = r2_score(y_test , test_pred)

print("Train Score (ETR):" , train_score_etr)
print("Test Score (ETR):" , test_score_etr)

In [ ]:
# test_df = test_df[X.columns] # To Prevent the Order as X
# test_preds_etr = etr_pipeline_model.predict(test_df)
# # test_preds_etr = np.clip(test_preds_etr, 0, None)
# print(test_preds_etr)

In [ ]:
# submission = pd.DataFrame({
#     'id': range(len(test_preds_etr)),  
#     'purchaseValue': test_preds_etr
# })

# # Save to CSV for submission
# submission.to_csv("submission.csv", index = False)

# # output in Kaggle notebook
# print("Submission file created successfully!")
# print(submission.head(10))

In [ ]:
param_dist_etr = {
    "et_reg__n_estimators" : randint(100 , 200),
    "et_reg__criterion" : ['squared_error'],
    "et_reg__max_depth" : randint(5 , 40), 
    "et_reg__max_features" : ['sqrt']
}

etr_search = RandomizedSearchCV(
    estimator = etr_pipeline,
    param_distributions = param_dist_etr,
    n_iter = 10,
    cv = 5,
    n_jobs = -1,
    scoring = 'r2',
    random_state = 42
)

etr_search.fit(X_train, y_train)

# using the best params from the HPT of the above code
best_etr = etr_search.best_estimator_

# Predicting
y_pred_test = best_etr.predict(X_test)
y_pred_train = best_etr.predict(X_train)

print("Best Params:", etr_search.best_params_)
print("Test R2:", r2_score(y_test, y_pred_test))
print("Train R2:", r2_score(y_train, y_pred_train))

# 5 mins

The ExtraTreesRegressor (ETR) gave strong performance even without tuning, with a high train R2 (~0.93) and a solid test R2 (~0.51), showing that the model could learn complex patterns quite well. After hyperparameter tuning, the scores remained almost the same, suggesting that the default settings were already working effectively. However, the slight gap between train and test scores indicates mild overfitting, as the model was learning a bit more from the training data than it could generalize to unseen data. This is expected from ETR, since it builds many deep trees on bootstrapped samples with full randomness , which makes it fast and accurate, but slightly prone to overfitting

In [ ]:
from sklearn.ensemble import VotingRegressor

model1 = XGBRegressor(max_depth = 8 , alpha = 2 , 
reg_lambda = 5 , subsample = 0.5683106657210144 , 
eta = 0.09358717652568162 , min_child_weight = 3 , 
gamma = 14 , colsample_bytree = 0.6270818245348694 , 
random_state = 42)

pipe = Pipeline([
    ('preprocessor', ct),
    ('model', model1)
])

pipe.fit(X_train, y_train)
r2_1 = pipe.score(X_test, y_test)

rf_pipeline_model.fit(X_train, y_train)
r2_2 = rf_pipeline_model.score(X_test, y_test)

etr_pipeline_model.fit(X_train, y_train)
r2_3 = etr_pipeline_model.score(X_test, y_test)

xgbrf_pipeline_model.fit(X_train , y_train)
r2_4 = xgbrf_pipeline_model.score(X_test , y_test)

total = r2_1 + r2_2 + r2_3 + r2_4
weights = [r2_1 / total, r2_2 / total, r2_3 / total , r2_4 / total]

print("Total sum of weights:", weights)

voting_reg = VotingRegressor(
    estimators = [
        ('xgb' , pipe),
        ('xgbrf' , xgbrf_pipeline_model),
        ('rf_reg' , rf_pipeline_model),
        ('et_reg' , etr_pipeline_model)
    ],
    weights = weights
)

voting_reg.fit(X_train, y_train)

y_train_pred = voting_reg.predict(X_train)
y_test_pred = voting_reg.predict(X_test)

print("Train R2:", r2_score(y_train, y_train_pred))
print("Test R2:", r2_score(y_test, y_test_pred))

# 5 mins

In [ ]:
test_preds_voting = voting_reg.predict(test_df)
test_preds_voting = np.clip(test_preds_voting, 0, None)
print(test_preds_voting)

In [ ]:
submission = pd.DataFrame({
    'id': range(len(test_preds_voting)),  
    'purchaseValue': test_preds_voting
})

# Save to CSV for submission
submission.to_csv("submission.csv", index = False)

# output in Kaggle notebook
print("Submission file created successfully!")
print(submission.head(10))

The Voting Regressor, which combined predictions from XGBoost, Random Forest, and ExtraTrees, gave the best overall performance among all models. Its test R2 score (~0.51) was the highest, and the train R2 (~0.89) was also strong, showing that the ensemble was able to balance between learning and generalization. By averaging the strengths of different models —> XGBoost’s boosting power, Random Forest’s robustness, and ETR’s speed and randomness as the Voting Regressor reduced individual model weaknesses and gave more stable predictions. Although slight overfitting may still be present due to a higher train score, the ensemble clearly helped improve the overall performance and reliability of predictions, including on the final test_df where it scored around 0.45, which was better than any single model.